# 02 - EXTRACTION DU FICHIER CSV DES MESURES RELEVEES

Dans ce notebook, nous allons extraire les fichiers CSV de notre fichier compressé et les enregistrer.

In [9]:
# Chargement des librairies
import requests
from pyspark.sql import SparkSession
import tarfile
from notebookutils import mssparkutils # La gestion du lakehouse
import os
from pathlib import Path

StatementMeta(, 2086e1c5-5ba9-429a-8737-dc123fdb531c, 11, Finished, Available, Finished, False)

## Définition des chemins d'extractions

In [6]:
# Répertoire temporaire local pour les fichiers CSV
TEMP_DIR = "/tmp/openradiation_extract"

# Destination finale dans le Lakehouse
LAKEHOUSE_DIR = "Files/OpenRadiation/extracted"

os.makedirs(TEMP_DIR, exist_ok=True)
mssparkutils.fs.mkdirs(LAKEHOUSE_DIR)

# Extraction des fichiers CSV uniquement
csv_files = []

StatementMeta(, 2086e1c5-5ba9-429a-8737-dc123fdb531c, 8, Finished, Available, Finished, False)

## Lecture et extraction du fichier compressé

In [7]:
# Chemin pour lire sera celui-ci
print(os.listdir("/lakehouse/default/Files/OpenRadiation/"))
ARCHIVE_PATH="/lakehouse/default/Files/OpenRadiation/openradiation_dataset.tar.gz"

StatementMeta(, 2086e1c5-5ba9-429a-8737-dc123fdb531c, 9, Finished, Available, Finished, False)

['extracted', 'openradiation_dataset.tar.gz']


In [10]:
# Ouverture du fichier compressé et récupération des fichiers CSV
with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
    for member in tar.getmembers():
        if not member.isfile():
            continue

        # Ne retenir que les CSV
        if not member.name.lower().endswith(".csv"):
            continue

        # Éviter les chemins dangereux dans l'archive
        filename = Path(member.name).name
        if not filename:
            continue

        source = tar.extractfile(member)
        if source is None:
            continue

        local_file = Path(TEMP_DIR) / filename

        with source, local_file.open("wb") as output:
            output.write(source.read())

        csv_files.append(local_file)

StatementMeta(, 2086e1c5-5ba9-429a-8737-dc123fdb531c, 12, Finished, Available, Finished, False)

## Enregistrement des fichiers CSV

In [11]:
# Copier les CSV vers le Lakehouse
for csv_file in csv_files:
    destination = f"{LAKEHOUSE_DIR}/{csv_file.name}"

    mssparkutils.fs.cp(
        f"file:{csv_file}",
        destination
    )

    print(f"Fichier extrait : {destination}")

print(f"\nNombre de fichiers CSV extraits : {len(csv_files)}")

StatementMeta(, 2086e1c5-5ba9-429a-8737-dc123fdb531c, 13, Finished, Available, Finished, False)

Fichier extrait : Files/OpenRadiation/extracted/measurements.csv
Fichier extrait : Files/OpenRadiation/extracted/tags.csv
Fichier extrait : Files/OpenRadiation/extracted/measurements_withoutEnclosedObject.csv

Nombre de fichiers CSV extraits : 3
